<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved. -->
<!-- SPDX-License-Identifier: Apache-2.0 -->

# Alpamayo 2 Edge: Continuous Open-Loop Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NVlabs/alpamayo2-edge/blob/main/notebooks/continuous_open_loop_evaluation.ipynb)

The [inference notebook](inference.ipynb) runs the model once, at t0 = 4.0 s of the example clip. This notebook runs it at every t0 across the same clip, every `STEP_SECONDS`, and measures each prediction against the path the vehicle actually drove:

1. **Find the evaluation window**: every t0 that has the model's full camera and ego history, and 6.4 s of logged future for scoring.
2. **Run the model at each t0** with the same inputs and sampling settings as the inference notebook, and time each model call.
3. **Score each prediction** against the logged future path, and compare it with a constant-velocity baseline.
4. **Plot errors over time** and bird's-eye views at several t0 values.
5. **Render an animation** of the front camera next to the latest plan. This step is optional.

This is **open-loop** evaluation: the vehicle always follows its logged path, and predictions are never fed back into the inputs. It measures how close each plan is to what the human driver did. It does not measure how the model would drive.

### Requirements

The same as the inference notebook: a CUDA GPU, a Hugging Face account with access to [nvidia/Alpamayo2-Edge](https://huggingface.co/nvidia/Alpamayo2-Edge) and to the [PhysicalAI-AV dataset](https://huggingface.co/datasets/nvidia/PhysicalAI-Autonomous-Vehicles), and this repository installed with the `notebook` extra:

```bash
uv sync --locked --extra notebook
source .venv/bin/activate
hf auth login
jupyter lab notebooks/continuous_open_loop_evaluation.ipynb
```

The run makes one model call per t0. The loop prints the time of each call, and the summary reports the peak GPU memory.

### Google Colab

1. Select **Runtime > Change runtime type > T4 GPU** (or a larger GPU).
2. Add a Hugging Face token with access to the model and the dataset as a Colab secret named `HF_TOKEN`, with notebook access turned on. Without the secret, the first cell asks for the token.
3. Run all cells. The first cell clones this repository, installs the pinned dependencies, and removes Colab's preinstalled PyTorch extensions, which do not match the pinned PyTorch; it does nothing outside Colab. If PyTorch was imported before the first cell ran, or Colab asks you to restart, select **Runtime > Restart session** and run all cells again.

On a Colab T4 (16 GB, Python 3.13, PyTorch 2.8.0, BF16, SDPA attention), the default 18-step run took 16 to 26 s per model call, about 6.5 minutes in total, with 9.8 GB peak GPU memory. Downloading the model and installing the dependencies took several more minutes.

In [ ]:
# Google Colab only: clone the repository and install the pinned notebook dependencies.
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    repo = Path("/content/alpamayo2-edge")
    if not repo.is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/NVlabs/alpamayo2-edge.git", repo],
            check=True,
        )
    os.chdir(repo)
    command = [sys.executable, "-m", "pip", "install", "--quiet", ".[notebook]"]
    if sys.version_info[:2] != (3, 12):
        # The project targets Python 3.12; install the pinned packages on Colab's Python anyway.
        command.append("--ignore-requires-python")
    subprocess.run(command, check=True)
    # Colab preinstalls PyTorch extensions built for its own PyTorch version. The locked
    # environment has none of them, and transformers imports them when they are installed.
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "--quiet", "--yes"]
        + ["torchaudio", "torchcodec", "torchao"],
        check=True,
    )

    from huggingface_hub import get_token, whoami

    if get_token() is None:
        from getpass import getpass

        from google.colab import userdata

        try:  # Colab asks for notebook access to the HF_TOKEN secret if it has not been granted.
            os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        except Exception:
            os.environ["HF_TOKEN"] = getpass("Hugging Face token (the input is hidden): ").strip()
    print(f"Hugging Face user: {whoami()['name']}")

In [ ]:
%matplotlib inline
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import physical_ai_av
import torch
import transformers

from alpamayo2_edge.checkpoint import load_model
from alpamayo2_edge.helper import prepare_model_inputs, to_device
from alpamayo2_edge.inference import load_clip_manifest, predict, validate_predictions
from alpamayo2_edge.load_physical_aiavdataset import load_physical_aiavdataset

In [ ]:
# A Hugging Face repository ID at the revision pinned in the README, or a local model directory.
MODEL_ID = "nvidia/Alpamayo2-Edge"
MODEL_REVISION = "f8951d095de3589b3face1dec3f4394c8ecedb38"
# A pinned PhysicalAI-AV dataset commit; None selects the latest commit.
DATASET_REVISION = "2ac2e9c056e0c8426face5bfad64b0961a295b28"
DEVICE = "cuda"
DTYPE = "bfloat16"
# PyTorch SDPA is the default; "flash_attention_2" also works when flash-attn is installed.
ATTN_IMPLEMENTATION = "sdpa"

# The model runs at t0, t0 + STEP_SECONDS, ... across the clip.
STEP_SECONDS = 1.0
# None evaluates the whole window; a small integer gives a quick check.
MAX_STEPS = None
# The inference notebook's settings: six candidates, each with its own reasoning and trajectory.
SETTINGS = dict(
    seed=43,
    top_p=0.98,
    top_k=None,
    temperature=0.6,
    num_traj_sets=6,
    num_traj_samples=1,
    max_generation_length=256,
    diffusion_kwargs={"inference_step": 10},
)
RENDER_ANIMATION = True
# Generated results; the repository's .gitignore excludes outputs/.
OUTPUT_DIR = Path("outputs/continuous_evaluation")

if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU.")

## Find the evaluation window

The notebook uses the clip and navigation instruction from `examples/nav_samples.json`, which the inference notebook also uses. The dataset does not include route instructions, so the same instruction, *"Turn right in 30m"*, is used at every t0 (see [Limitations](#Limitations)).

A t0 is valid when the dataset has everything the model reads and everything needed to score it:
- **Camera history**: 2 s before t0 for the front wide and front telephoto cameras, and 1 s for the other four, as `load_physical_aiavdataset` requests.
- **Ego history**: 1.9 s of egomotion before t0.
- **Logged future**: 6.4 s of egomotion after t0, the prediction horizon. The model does not see it.

The window starts at the first valid t0, rounded up to a 0.1 s boundary.

In [ ]:
candidates = [Path("examples/nav_samples.json"), Path("../examples/nav_samples.json")]
manifest = next((path for path in candidates if path.is_file()), None)
if manifest is None:
    raise FileNotFoundError("Keep this notebook in the repository's notebooks/ directory.")
example = load_clip_manifest(manifest, index=0)  # clip_id, t0_us and nav_text
CLIP_ID, NAV_TEXT = example["clip_id"], example["nav_text"]

avdi = physical_ai_av.PhysicalAIAVDatasetInterface(revision=DATASET_REVISION)
egomotion = avdi.get_clip_feature(CLIP_ID, avdi.features.LABELS.EGOMOTION, maybe_stream=True)

HORIZON_STEPS, DT_US = 64, 100_000  # 64 future poses at 0.1 s intervals
HORIZON_US = HORIZON_STEPS * DT_US
STEP_US = round(STEP_SECONDS * 1e6)
# Image history before t0 that load_physical_aiavdataset requests from each camera.
CAMERA_HISTORY_US = {
    "CAMERA_CROSS_LEFT_120FOV": 1_000_000,
    "CAMERA_FRONT_WIDE_120FOV": 2_000_000,
    "CAMERA_CROSS_RIGHT_120FOV": 1_000_000,
    "CAMERA_REAR_LEFT_70FOV": 1_000_000,
    "CAMERA_REAR_RIGHT_70FOV": 1_000_000,
    "CAMERA_FRONT_TELE_30FOV": 2_000_000,
}

ego_start_us, ego_end_us = (int(t) for t in egomotion.time_range)
first_t0 = [2_000_000, ego_start_us + 1_900_000]
last_t0 = [ego_end_us - HORIZON_US]
cameras = {}
for name, history_us in CAMERA_HISTORY_US.items():
    cameras[name] = avdi.get_clip_feature(
        CLIP_ID, getattr(avdi.features.CAMERA, name), maybe_stream=True
    )
    timestamps = np.asarray(cameras[name].timestamps)
    first_t0.append(int(timestamps[0]) + history_us)
    last_t0.append(int(timestamps[-1]))
start_us = -(-max(first_t0) // DT_US) * DT_US  # round up to a 0.1 s boundary
end_us = min(last_t0)
t0_values = np.arange(start_us, end_us + 1, STEP_US, dtype=np.int64)
if MAX_STEPS is not None:
    t0_values = t0_values[:MAX_STEPS]
if len(t0_values) == 0:
    raise ValueError("The clip is too short for the camera history and the 6.4 s horizon.")

print(f"Clip {CLIP_ID}, navigation instruction: {NAV_TEXT!r}")
print(f"Egomotion: {ego_start_us / 1e6:.1f} s to {ego_end_us / 1e6:.1f} s")
print(
    f"Evaluation window: t0 = {t0_values[0] / 1e6:.1f} s to {t0_values[-1] / 1e6:.1f} s, "
    f"{len(t0_values)} steps of {STEP_SECONDS:g} s"
)

## Metrics

At each t0, the logged future path is read from egomotion and transformed into the ego frame at t0 (x forward, y left), the frame of the model's predictions and the frame the inference notebook plots. Errors are planar (x, y) distances in metres:

- **ADE**, the average displacement error over the 64 future poses, and **FDE**, the final displacement error at 6.4 s.
- **L2 at 1 s and 3 s**: the displacement error at those times.

With six candidates per t0, each metric is reported two ways:
- **Average candidate**: the mean over the six candidates, the expected error of one sampled plan.
- **Best of six**: the lowest error among the six (minADE and minFDE). This selects a candidate with the logged path, which the model does not see, so it measures how well the candidates cover the logged path.

The **constant-velocity baseline** extends the vehicle's average velocity over the last 0.5 s of ego history in a straight line. It uses only the model's ego history input, and gives a reference for each error.

In [ ]:
DT_S = DT_US / 1e6
L2_HORIZONS_S = (1.0, 3.0)


def logged_future(t0_us):
    """Logged positions at t0 + 0.1 s, ..., t0 + 6.4 s in the ego frame at t0: [64, 3]."""
    times = t0_us + np.arange(HORIZON_STEPS + 1, dtype=np.int64) * DT_US
    poses = egomotion(times).pose
    positions = np.asarray(poses.translation)
    return poses.rotation[0].inv().apply(positions - positions[0])[1:]


def constant_velocity(history_xyz, window_steps=5):
    """Extend the average velocity of the last window_steps * 0.1 s of history: [64, 3]."""
    velocity = (history_xyz[-1] - history_xyz[-1 - window_steps]) / (window_steps * DT_S)
    seconds = DT_S * np.arange(1, HORIZON_STEPS + 1)[:, None]
    return history_xyz[-1] + seconds * velocity


def displacement_errors(predicted, logged):
    """Planar distances for each trajectory and future pose: [trajectories, 64]."""
    return np.linalg.norm(predicted[..., :2] - logged[None, :, :2], axis=-1)


def score(errors, prefix, reduce):
    """ADE, FDE and L2 at L2_HORIZONS_S, reduced over trajectories with reduce."""
    row = {f"{prefix} ADE": reduce(errors.mean(axis=1)), f"{prefix} FDE": reduce(errors[:, -1])}
    for seconds in L2_HORIZONS_S:
        row[f"{prefix} L2@{seconds:g}s"] = reduce(errors[:, round(seconds / DT_S) - 1])
    return row

## Load the model

`load_model` downloads the checkpoint at the pinned revision (about 5.3 GB) and rejects incomplete or mismatched weights, as in the inference notebook.

In [ ]:
model, _ = load_model(
    MODEL_ID,
    device=DEVICE,
    dtype=DTYPE,
    attn_implementation=ATTN_IMPLEMENTATION,
    revision=MODEL_REVISION,  # used only for Hugging Face repository IDs
)
print(f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B parameters on {model.device}")

## Run the model across the clip

At each t0, `load_physical_aiavdataset` reads the six cameras and the ego history, and `predict` runs the model with `SETTINGS`, as in the inference notebook. The loop records two times separately:
- **Model call**: `predict` alone, between CUDA synchronizations.
- **Data loading**: streaming and decoding the camera frames, and building the prompt.

Interrupting the kernel stops the loop; the cells below then use the steps that finished.

In [ ]:
rows, records = [], []
torch.cuda.reset_peak_memory_stats(DEVICE)
try:
    for t0_us in t0_values.tolist():
        load_start = time.perf_counter()
        data = load_physical_aiavdataset(CLIP_ID, t0_us, nav_text=NAV_TEXT, avdi=avdi)
        inputs = to_device(prepare_model_inputs(data, model.config, model.tokenizer), device=DEVICE)
        load_seconds = time.perf_counter() - load_start

        torch.cuda.synchronize(DEVICE)
        model_start = time.perf_counter()
        pred_xyz, pred_rot, extra = predict(model, inputs, SETTINGS, DEVICE)
        torch.cuda.synchronize(DEVICE)
        model_seconds = time.perf_counter() - model_start
        validate_predictions(
            pred_xyz, pred_rot, SETTINGS["num_traj_sets"], SETTINGS["num_traj_samples"]
        )

        predicted = pred_xyz[0].reshape(-1, HORIZON_STEPS, 3).float().cpu().numpy()
        history = data["ego_history_xyz"][0].numpy()  # [20, 3] from -1.9 s to t0
        logged = logged_future(t0_us)
        baseline = constant_velocity(history)
        model_errors = displacement_errors(predicted, logged)  # [candidates, 64]
        baseline_errors = displacement_errors(baseline[None], logged)  # [1, 64]
        speed = float(np.linalg.norm(egomotion(np.array([t0_us])).velocity[0, :2]))

        rows.append(
            {
                "t0 (s)": t0_us / 1e6,
                "speed (m/s)": speed,
                **score(model_errors, "best of six", np.min),
                **score(model_errors, "average candidate", np.mean),
                **score(baseline_errors, "constant velocity", np.mean),
                "model call (s)": model_seconds,
                "data loading (s)": load_seconds,
            }
        )
        records.append(
            {
                "t0_us": t0_us,
                "predicted": predicted,
                "history": history,
                "logged": logged,
                "baseline": baseline,
                "candidate_ade": model_errors.mean(axis=1),
                "cot": [str(text) for text in np.asarray(extra["cot"]).reshape(-1)],
            }
        )
        row = rows[-1]
        print(
            f"t0 = {row['t0 (s)']:5.1f} s | {speed:4.1f} m/s | ADE: average candidate "
            f"{row['average candidate ADE']:5.2f} m, best of six {row['best of six ADE']:5.2f} m, "
            f"constant velocity {row['constant velocity ADE']:5.2f} m | "
            f"model call {model_seconds:5.1f} s"
        )
except KeyboardInterrupt:
    print(f"Stopped after {len(records)} of {len(t0_values)} steps.")
if not records:
    raise RuntimeError("No steps finished.")
peak_memory_gb = torch.cuda.max_memory_allocated(DEVICE) / 1e9

## Summary

Means over the evaluated t0 values. The run details record the GPU, library versions and settings, and the cell saves the metrics, the trajectories and the reasoning to `OUTPUT_DIR`.

In [ ]:
metrics = pd.DataFrame(rows).set_index("t0 (s)")
methods = ("average candidate", "best of six", "constant velocity")
summary = pd.DataFrame(
    {
        method: [metrics[f"{method} {name}"].mean() for name in ("ADE", "FDE", "L2@1s", "L2@3s")]
        for method in methods
    },
    index=["ADE (m)", "FDE (m)", "L2 at 1 s (m)", "L2 at 3 s (m)"],
)
run = {
    "gpu": torch.cuda.get_device_name(DEVICE),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "dtype": DTYPE,
    "attention": ATTN_IMPLEMENTATION,
    "model": {"id": MODEL_ID, "revision": MODEL_REVISION},
    "dataset_revision": DATASET_REVISION,
    "clip_id": CLIP_ID,
    "nav_text": NAV_TEXT,
    "settings": SETTINGS,
    "step_seconds": STEP_SECONDS,
    "steps": len(records),
    "model_call_seconds": {
        "median": float(metrics["model call (s)"].median()),
        "max": float(metrics["model call (s)"].max()),
    },
    "peak_gpu_memory_gb": round(peak_memory_gb, 2),
    "summary": summary.round(3).to_dict(),
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metrics.to_csv(OUTPUT_DIR / "metrics.csv")
(OUTPUT_DIR / "run.json").write_text(json.dumps(run, indent=2) + "\n")
(OUTPUT_DIR / "reasoning.json").write_text(
    json.dumps({str(r["t0_us"]): r["cot"] for r in records}, indent=2) + "\n"
)
np.savez_compressed(
    OUTPUT_DIR / "trajectories.npz",
    t0_us=np.array([r["t0_us"] for r in records]),
    **{
        key: np.stack([r[key] for r in records])
        for key in ("predicted", "history", "logged", "baseline")
    },
)

print(
    f"{run['gpu']}, {DTYPE}, torch {run['torch']}, transformers {run['transformers']}: "
    f"{len(records)} steps, median model call {run['model_call_seconds']['median']:.1f} s, "
    f"peak GPU memory {peak_memory_gb:.1f} GB"
)
print(f"Saved metrics, trajectories and reasoning to {OUTPUT_DIR.resolve()}")
summary.round(2)

## Errors over time

The top two panels show ADE and FDE at each t0, and the bottom panel shows the logged speed. The dotted vertical line marks t0 = 4.0 s, the single time step the inference notebook evaluates.

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
STYLES = {
    "average candidate": dict(color=PALETTE[0], marker="o"),
    "best of six": dict(color=PALETTE[2], marker="o"),
    "constant velocity": dict(color="#52514e", marker="s", ls="--"),
}

fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True, height_ratios=[2, 2, 1])
for ax, name in zip(axes[:2], ("ADE", "FDE")):
    for method, style in STYLES.items():
        ax.plot(metrics.index, metrics[f"{method} {name}"], lw=2, ms=4, label=method, **style)
    ax.set_ylabel(f"{name} (m)")
    ax.set_ylim(bottom=0)
axes[2].plot(metrics.index, metrics["speed (m/s)"], color=PALETTE[1], lw=2)
axes[2].set_ylabel("speed (m/s)")
axes[2].set_xlabel("t0 (s)")
for ax in axes:
    ax.axvline(example["t0_us"] / 1e6, color="#898781", ls=":", lw=1.5)
    ax.grid(color="#e1e0d9", lw=0.8)
axes[0].legend(loc="upper left", fontsize=9)
axes[0].set_title(f"Open-loop errors across clip {CLIP_ID[:8]}, navigation: {NAV_TEXT!r}")
fig.tight_layout()
plt.show()

## Bird's-eye views

Up to six evenly spaced t0 values, drawn like the inference notebook's bird's-eye view (forward up, left on the left). Each panel shows the six candidates, the ego history, the logged future path (dashed) and the constant-velocity baseline (dotted). Below the plots, the reasoning of the candidate closest to the logged path is printed for each panel. Some candidates return `<NOMINAL>` instead of reasoning text; in this clip, that happens mostly on the straight segments before and after the intersection.

In [ ]:
def draw_plan(ax, record, show_legend=False):
    """Draw one t0's candidates, history, logged path and baseline in the ego frame at t0."""
    history, logged, baseline = record["history"], record["logged"], record["baseline"]
    ax.plot(history[:, 1], history[:, 0], "-.", color="#52514e", lw=2, label="Ego history")
    for k, xyz in enumerate(record["predicted"]):
        ax.plot(
            xyz[:, 1],
            xyz[:, 0],
            color=PALETTE[k % len(PALETTE)],
            lw=2,
            label=f"Candidate {k + 1}",
        )
    ax.plot(logged[:, 1], logged[:, 0], "--", color="#0b0b0b", lw=2, label="Logged future path")
    ax.plot(baseline[:, 1], baseline[:, 0], ":", color="#898781", lw=2, label="Constant velocity")
    ax.plot(0, 0, "^", color="#0b0b0b", ms=9)
    ax.set_aspect("equal", adjustable="datalim")
    ax.grid(color="#e1e0d9", lw=0.8)
    if show_legend:
        ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=9)


shown = np.unique(np.linspace(0, len(records) - 1, min(6, len(records))).round().astype(int))
fig, axes = plt.subplots(1, len(shown), figsize=(3.2 * len(shown) + 2, 6.5), squeeze=False)
for ax, index in zip(axes[0], shown):
    record = records[index]
    draw_plan(ax, record, show_legend=ax is axes[0, len(shown) - 1])
    ax.invert_xaxis()  # Positive y (left) appears on the left.
    ax.set_title(f"t0 = {record['t0_us'] / 1e6:.1f} s", fontsize=10)
    ax.set_xlabel("y (m), left")
axes[0, 0].set_ylabel("x (m), forward")
fig.tight_layout()
plt.show()

for index in shown:
    record = records[index]
    best = int(np.argmin(record["candidate_ade"]))
    print(
        f"t0 = {record['t0_us'] / 1e6:.1f} s, candidate {best + 1} "
        f"(ADE {record['candidate_ade'][best]:.2f} m): {record['cot'][best]}"
    )

## Optional: animation

Set `RENDER_ANIMATION = False` to skip this step. Each frame shows the front wide camera image at 10 frames per second next to the most recent plan. The plan stays in the ego frame of its t0, and the black dot marks where the vehicle actually is at the frame's time, so the dot moves along the logged path until the next t0 replaces the plan. Plans are not projected into the camera image.

The animation is saved to `OUTPUT_DIR` as MP4 when FFmpeg is installed, and as a GIF otherwise.

In [ ]:
if RENDER_ANIMATION:
    from matplotlib import animation

    front = cameras["CAMERA_FRONT_WIDE_120FOV"]
    last_frame_us = int(np.asarray(front.timestamps)[-1])
    if animation.FFMpegWriter.isAvailable():
        writer, path = animation.FFMpegWriter(fps=10), OUTPUT_DIR / "continuous_evaluation.mp4"
    else:
        writer, path = animation.PillowWriter(fps=10), OUTPUT_DIR / "continuous_evaluation.gif"

    # Fixed bird's-eye-view limits so the axes do not jump between plans.
    points = np.concatenate(
        [r["predicted"][..., :2].reshape(-1, 2) for r in records]
        + [r["logged"][:, :2] for r in records]
    )
    forward_max = max(10.0, float(points[:, 0].max()) + 5)
    lateral_max = max(10.0, float(np.abs(points[:, 1]).max()) + 5)

    fig, (image_ax, bev_ax) = plt.subplots(1, 2, figsize=(13, 5.4), width_ratios=[1.7, 1])
    frames = 0
    with writer.saving(fig, str(path), dpi=100):
        for record in records:
            t0_us = record["t0_us"]
            times = np.arange(t0_us, min(t0_us + STEP_US, last_frame_us + 1), DT_US)
            images, image_times = front.decode_images_from_timestamps(times)
            image_times = np.asarray(image_times, dtype=np.int64)
            poses = egomotion(np.concatenate([[t0_us], image_times])).pose
            positions = np.asarray(poses.translation)
            current = poses.rotation[0].inv().apply(positions - positions[0])[1:]
            for image, frame_us, position in zip(images, image_times, current):
                image_ax.clear()
                image_ax.imshow(np.asarray(image))
                image_ax.set_axis_off()
                image_ax.set_title(f"Front wide camera, t = {frame_us / 1e6:.1f} s", fontsize=11)
                bev_ax.clear()
                draw_plan(bev_ax, record)
                bev_ax.set_aspect("equal", adjustable="box")  # keep the fixed limits below
                bev_ax.plot(position[1], position[0], "o", color="#0b0b0b", ms=8)
                bev_ax.set_xlim(lateral_max, -lateral_max)  # Positive y (left) on the left.
                bev_ax.set_ylim(-12, forward_max)
                bev_ax.set_xlabel("y (m), left")
                bev_ax.set_ylabel("x (m), forward")
                bev_ax.set_title(f"Plan from t0 = {t0_us / 1e6:.1f} s", fontsize=11)
                fig.suptitle(
                    f"Alpamayo 2 Edge, open-loop replay, navigation: {NAV_TEXT!r}", fontsize=12
                )
                writer.grab_frame()
                frames += 1
    plt.close(fig)
    print(f"Saved {frames} frames to {path.resolve()}")

## Limitations

- **Open loop**: the inputs at every t0 come from the logged drive, so errors do not compound and the metrics do not show how the model would drive on its own.
- **Fixed navigation instruction**: the dataset has no route instructions, so every t0 uses *"Turn right in 30m"*. The distance in the instruction is wrong at most t0 values, and the instruction no longer applies after the turn. Errors at those t0 values partly reflect the instruction rather than the model.
- **One clip and one sample per setting**: the results come from a single clip with fixed seeds, and sampled reasoning and trajectories can change with the GPU, attention backend and library versions. They are not a benchmark.
- **Simple baseline**: the constant-velocity baseline cannot turn, so it is weakest in turns. It gives a reference for each error, not a competing planner.